## Text input

In [10]:
from dotenv import load_dotenv
from pprint import pprint
from langchain.agents import create_agent
from langchain.messages import HumanMessage

load_dotenv()

True

In [4]:
agent = create_agent(
    model='gpt-5-nano',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [ ]:
question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

## Image input

In [16]:
from ipywidgets import FileUpload
from IPython.display import display
import base64

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [17]:
pprint(type(uploader.value))
pprint(uploader.value)

<class 'tuple'>
({'content': <memory at 0x112dca2c0>,
  'last_modified': datetime.datetime(2026, 4, 8, 20, 6, 52, 3000, tzinfo=datetime.timezone.utc),
  'name': 'city.png',
  'size': 205750,
  'type': 'image/png'},)


In [18]:
# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes: bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [19]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this view"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

What you’re looking at is a hinge between old and new, a capital city built to honor its past while racing toward the future. The red-brick church with twin white spires is the Heart Gate Cathedral, a centuries-old beacon around which the city grew. Its rounded apse and tiled roof give it a medieval, almost fortress-like dignity, but the slender spires reach skyward with a quiet, modern confidence. The square around it—the Circle of Echoes, you could call it—gathers people, vendors, cyclists, and the occasional drone parade as if to remind the city that faith, ceremony, and daily life still share the same stage.

Beyond the church, the skyline unfurls in a telltale blend of eras. Trees line the avenues, their canopies a green halo that softens the glass and steel of the new districts. In the distance, tall glass towers rise like punctuation marks: some with sleek, minimalist silhouettes, others wrapped in latticework or solar skins that shimmer with the day’s light. There’s a sense of 

## Audio input

In [21]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.52it/s]


Done.


In [23]:
agent = create_agent(
    model='gpt-4o-audio-preview',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    input={"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

The audio you provided is a request for a poem about love. Based on that, here is a heartfelt poem:

Love is the whisper of the morning breeze,
A gentle touch among ancient trees.
It’s the laughter shared in quiet hours,
The bond that blooms like summer flowers.

It’s the patience when the road runs long,
The melody of an old sweet song.
It’s the joy when shadows start to flee,
The steady hand through mystery.

Love is the courage in a fragile heart,
A flame that burns when worlds fall apart.
It’s the light that glows through darkest night,
Guiding souls with silent, tender light.
